# 🧠 CerebroSpheroid V5: High-Performance Colab Training & Cloud Sync
### Multi-Model Sync Hub (Small, Compact, Base) & Dataset Mixture Selector

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Colab_Training.ipynb)

**Key Highlights:**
- **3 Model Tiers Supported:** `v5_small` (101M, 16 shells), `v5_compact` (244M, 32 shells), and `v5_base` (617M, 48 shells).
- **Two-Way Multi-Model Cloud Sync:** Each model tier automatically synchronizes its own `{model}_latest.pt` and `{model}_best.pt` with `https://bioflo.in/api/cerebro/sync.php`.
- **Dedicated Dataset Mixtures:** Launch focused training on Foundation, Math, Code, Agentic Tools, or Golden Consolidation Replay from dedicated cells.
- **V4 Elastic Sliding Context Window:** Zero padding waste, complementary $(L_p, L_t)$ buckets, and preceding context feedback conditioning.

## 1. Verify GPU Hardware Acceleration & System Environment

# Optional: Authenticate HuggingFace API if HF_TOKEN is stored in Colab Secrets
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        import os
        os.environ["HF_TOKEN"] = hf_token
        print("✓ Successfully loaded HF_TOKEN from Colab Secrets")
except Exception:
    pass


In [ ]:
import torch, os, sys
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Running on CPU! Go to Runtime -> Change runtime type -> Select T4 GPU or A100 for fast training.")

!nvidia-smi

## 2. Clone CerebroSpheroid V5 Repository & Install Dependencies

In [ ]:
%cd /content
if not os.path.exists("/content/cerebro_spheroid_v5"):
    !git clone https://github.com/AjitKumar-git/cerebro_spheroid_v5.git /content/cerebro_spheroid_v5
else:
    %cd /content/cerebro_spheroid_v5
    !git fetch --all && git reset --hard origin/main

%cd /content/cerebro_spheroid_v5

# Install minimal fast training prerequisites
!pip install -q transformers datasets accelerate sentencepiece pyarrow

## 3. Select Model Architecture Tier
Choose which CerebroSpheroid model tier to train or resume:
- **`v5_small` (101.6M params, 16 shells)**: Native 16-shell geometry for high-throughput training.
- **`v5_compact` (244.4M params, 32 shells)**: High-capacity multi-shell cognitive model.
- **`v5_base` (617.3M params, 48 shells)**: Frontier-scale capacity.

In [ ]:
#@title ⚙️ Unified Training & Workflow Configuration
#@markdown Select model tier, execution mode, and hyper-parameters. When clicking **"Run All"**, cells will automatically skip based on checkpoint step progress!

MODEL_PRESET = "v5_small" #@param ["v5_small", "v5_compact", "v5_base"]
TRAINING_MODE = "Step-Aware Progressive Stages (5B -> 5C -> 5D -> 5E -> 5F)" #@param ["Step-Aware Progressive Stages (5B -> 5C -> 5D -> 5E -> 5F)", "Single Continuous Run (Option 5A)"]

BATCH_SIZE = 16 #@param {type:"integer"}
GRAD_ACCUM = 4 #@param {type:"integer"}
PROMPT_LEN = 32 #@param {type:"integer"}
SEQ_LEN = 512 #@param {type:"integer"}
TOTAL_WINDOW = 544 #@param {type:"integer"}

#@markdown **Mixed Precision Precision:**
AMP_DTYPE = "auto" #@param ["auto", "float16", "bfloat16", "float32"]

#@markdown **Deliberation & Self-Referential Reasoning:**
DELIBERATION_MODE = "teacher_forcing" #@param ["teacher_forcing", "scheduled", "pitfall_recovery", "internal"]
SCHEDULE_STEPS = 0 #@param {type:"integer"}
MIN_SAMPLING_P = 1.00 #@param {type:"number"}
PITFALL_PROB = 0.0 #@param {type:"number"}

#@markdown **Dynamic Loss-Window Governor (Clamps loss within [2.80, 3.25]):**
LOSS_WINDOW_GOVERNOR = True #@param {type:"boolean"}
LOSS_CEILING = 3.25 #@param {type:"number"}
LOSS_FLOOR = 2.80 #@param {type:"number"}
LOSS_LR_DAMPING = True #@param {type:"boolean"}

#@markdown **Phase Target Step Thresholds (Cells automatically skip if checkpoint step >= target):**
PHASE_1_TARGET = 50000 #@param {type:"integer"}
PHASE_2_TARGET = 200000 #@param {type:"integer"}
PHASE_3_TARGET = 400000 #@param {type:"integer"}
PHASE_4_TARGET = 700000 #@param {type:"integer"}
PHASE_5_TARGET = 1000000 #@param {type:"integer"}

print("=" * 60)
print(f"✓ Active Model Tier  : {MODEL_PRESET}")
print(f"✓ Execution Mode     : {TRAINING_MODE}")
print(f"✓ Micro Batch Size   : {BATCH_SIZE}")
print(f"✓ Gradient Accum     : {GRAD_ACCUM}")
print(f"✓ Effective Batch    : {BATCH_SIZE * GRAD_ACCUM} prompts/step ({BATCH_SIZE * GRAD_ACCUM * TOTAL_WINDOW:,} tokens/step)")
print(f"✓ Context Window     : Prompt={PROMPT_LEN}, Target={SEQ_LEN} (Total={TOTAL_WINDOW})")
print(f"✓ Deliberation Mode  : {DELIBERATION_MODE} (floor={MIN_SAMPLING_P}, pitfall={PITFALL_PROB})")
print(f"✓ Loss Governor      : [{LOSS_FLOOR:.2f}, {LOSS_CEILING:.2f}] active={LOSS_WINDOW_GOVERNOR}")
print("=" * 60)


## 4. Multi-Model Cloud Hub Status & Checkpoint Resumption
Queries `https://bioflo.in/api/cerebro/sync.php` for available checkpoints for this model tier (`latest` and `best`).

In [ ]:
from src.cloud_sync import check_remote_manifest, pull_checkpoint
import os, torch

# Check remote manifest on bioflo.in
norm_model = "v5_small" if MODEL_PRESET in ("tpu_v5e_1", "tpu_v6e_1", "v5_small") else MODEL_PRESET
manifest = check_remote_manifest(model=norm_model)
print(f"Sync Hub Online: {manifest.get('status') == 'online'!s}")

# Pull latest checkpoint from cloud hub if available
local_ckpt_dir = f"checkpoints/{MODEL_PRESET}"
os.makedirs(local_ckpt_dir, exist_ok=True)
target_file = f"{local_ckpt_dir}/{MODEL_PRESET}_latest.pt"

m_data = manifest.get("manifest", {})
models_block = manifest.get("models", {})
m_info = models_block.get(norm_model, {})

# Select the most advanced slot between model-specific entry and root fallback
cand_latest = [
    e for e in [m_data.get(f"{norm_model}_latest"), m_info.get("latest_manifest"), m_data.get("latest")]
    if isinstance(e, dict) and "step" in e
]
latest_entry = max(cand_latest, key=lambda x: int(x.get("step", 0))) if cand_latest else {}

cand_best = [
    e for e in [m_data.get(f"{norm_model}_best"), m_info.get("best_manifest"), m_data.get("best")]
    if isinstance(e, dict) and "step" in e
]
best_entry = min(cand_best, key=lambda x: float(x.get("loss", float("inf")))) if cand_best else {}

latest_step = latest_entry.get("step", "None")
latest_loss = latest_entry.get("loss", "None")
best_step = best_entry.get("step", "None")
best_loss = best_entry.get("loss", "None")

if latest_step != "None":
    print(f"   Latest remote step: {int(latest_step):,} (loss: {float(latest_loss):.6f})")
    pull_checkpoint(target_file, target_type="latest", model=norm_model)
else:
    print("   Latest remote step: None")
if best_step != "None":
    print(f"   Best remote step  : {int(best_step):,} (loss: {float(best_loss):.6f})")
else:
    print("   Best remote step  : None")

def get_current_model_step(model_preset):
    """Inspects local checkpoints to detect the current trained step."""
    candidates = [
        f"checkpoints/{model_preset}/{model_preset}_latest.pt",
        f"checkpoints/{model_preset}/tpu_v5e_1_latest.pt",
        f"checkpoints/{model_preset}/latest.pt",
        f"checkpoints/{model_preset}/{model_preset}_best.pt",
        f"checkpoints/{model_preset}/{model_preset}_graded_final.pt",
    ]
    for p in candidates:
        if os.path.exists(p):
            try:
                ckpt = torch.load(p, map_location="cpu", weights_only=False)
                step = int(ckpt.get("step", 0))
                return step, p
            except Exception:
                continue
    return 0, None

curr_step, ckpt_loc = get_current_model_step(MODEL_PRESET)
print(f"
✓ Active Checkpoint   : {ckpt_loc if ckpt_loc else 'None (Starting Fresh from Step 0)'}")
print(f"✓ Current Model Step  : {curr_step:,}")


## 5. Training by Dataset Mixture
Select and run one of the cells below depending on your training objective.

### 🔄 Option 5A: Full Graded Developmental Curriculum (`--mix auto`)
Trains across the complete 4-phase Warmup-Stable-Decay (WSD) developmental schedule.

In [ ]:
# Option 5A: Single Continuous Run (--mix auto)
curr_step, _ = get_current_model_step(MODEL_PRESET)

if TRAINING_MODE.startswith("Step-Aware"):
    print("⏩ [Option 5A] Skipped: 'Step-Aware Progressive Stages' is active.")
    print("✓ Proceeding automatically to staged curriculum cells (5B -> 5F)...")
elif curr_step >= PHASE_5_TARGET:
    print(f"⏩ [Option 5A] Model already reached {curr_step:,} >= {PHASE_5_TARGET:,}. Skipping cell.")
else:
    print(f"🚀 [Option 5A] Launching continuous run from step {curr_step:,} -> {PHASE_5_TARGET:,}...")
    !python3 scripts/train_v5_graded_curriculum.py --model-preset {MODEL_PRESET} --mix auto --total-steps {PHASE_5_TARGET} --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --prompt-len {PROMPT_LEN} --seq-len {SEQ_LEN} --no-elastic-window --max-lr 5.0e-4 --warmup-steps 500 --save-interval 250 --cloud-sync --cloud-sync-interval 100 --amp-dtype {AMP_DTYPE} --optimizer spheroid --deliberation-mode {DELIBERATION_MODE} --schedule-steps {SCHEDULE_STEPS} --min-sampling-p {MIN_SAMPLING_P} --pitfall-prob {PITFALL_PROB} --loss-window-governor --loss-ceiling {LOSS_CEILING} --loss-floor {LOSS_FLOOR} --loss-lr-damping


### 📚 Option 5B: Mixture 1 - Grammar, Lexical Syntax & Educational Prose (`--mix foundation`)
High-density Cosmopedia v2 + FineWeb-Edu + Clean Wikipedia. Grounds vocabulary and syntax.

In [ ]:
# Option 5B: Phase 1 - Foundation & Grammar (Target: PHASE_1_TARGET)
curr_step, _ = get_current_model_step(MODEL_PRESET)

if not TRAINING_MODE.startswith("Step-Aware"):
    print("⏩ [Option 5B] Skipped: Single Continuous Run mode is active.")
elif curr_step >= PHASE_1_TARGET:
    print(f"⏩ [Option 5B: Foundation] Step {curr_step:,} >= target {PHASE_1_TARGET:,}. Stage already complete!")
    print("✓ Skipping cell to proceed to next stage...")
else:
    print(f"🚀 [Option 5B: Foundation] Launching training from step {curr_step:,} -> {PHASE_1_TARGET:,}...")
    !python3 scripts/train_v5_graded_curriculum.py --model-preset {MODEL_PRESET} --mix foundation --total-steps {PHASE_1_TARGET} --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --prompt-len {PROMPT_LEN} --seq-len {SEQ_LEN} --no-elastic-window --max-lr 5.0e-4 --warmup-steps 500 --save-interval 250 --cloud-sync --cloud-sync-interval 100 --amp-dtype {AMP_DTYPE} --optimizer spheroid --deliberation-mode {DELIBERATION_MODE} --schedule-steps {SCHEDULE_STEPS} --min-sampling-p {MIN_SAMPLING_P} --pitfall-prob {PITFALL_PROB} --loss-window-governor --loss-ceiling {LOSS_CEILING} --loss-floor {LOSS_FLOOR} --loss-lr-damping


### 🛠️ Option 5C: Mixture 2 - Multi-Step Mathematics & Deductive Reasoning (`--mix reasoning`)
GSM8K + Open-Web-Math + OpenHermes-2.5 multi-step Chain-of-Thought.

In [ ]:
# Option 5C: Phase 2 - Reasoning & Mathematics (Target: PHASE_2_TARGET)
curr_step, _ = get_current_model_step(MODEL_PRESET)

if not TRAINING_MODE.startswith("Step-Aware"):
    print("⏩ [Option 5C] Skipped: Single Continuous Run mode is active.")
elif curr_step >= PHASE_2_TARGET:
    print(f"⏩ [Option 5C: Reasoning] Step {curr_step:,} >= target {PHASE_2_TARGET:,}. Stage already complete!")
    print("✓ Skipping cell to proceed to next stage...")
else:
    print(f"🚀 [Option 5C: Reasoning] Launching training from step {curr_step:,} -> {PHASE_2_TARGET:,}...")
    !python3 scripts/train_v5_graded_curriculum.py --model-preset {MODEL_PRESET} --mix reasoning --total-steps {PHASE_2_TARGET} --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --prompt-len {PROMPT_LEN} --seq-len {SEQ_LEN} --no-elastic-window --max-lr 3.5e-4 --warmup-steps 500 --save-interval 250 --cloud-sync --cloud-sync-interval 100 --amp-dtype {AMP_DTYPE} --optimizer spheroid --deliberation-mode {DELIBERATION_MODE} --schedule-steps {SCHEDULE_STEPS} --min-sampling-p {MIN_SAMPLING_P} --pitfall-prob {PITFALL_PROB} --loss-window-governor --loss-ceiling {LOSS_CEILING} --loss-floor {LOSS_FLOOR} --loss-lr-damping


### 💻 Option 5D: Mixture 3 - Python Code, Algorithms & Unit Tests (`--mix code`)
The Stack Python-Edu + Algorithmic derivations.

In [ ]:
# Option 5D: Phase 3 - Python Code & Algorithms (Target: PHASE_3_TARGET)
curr_step, _ = get_current_model_step(MODEL_PRESET)

if not TRAINING_MODE.startswith("Step-Aware"):
    print("⏩ [Option 5D] Skipped: Single Continuous Run mode is active.")
elif curr_step >= PHASE_3_TARGET:
    print(f"⏩ [Option 5D: Code] Step {curr_step:,} >= target {PHASE_3_TARGET:,}. Stage already complete!")
    print("✓ Skipping cell to proceed to next stage...")
else:
    print(f"🚀 [Option 5D: Code] Launching training from step {curr_step:,} -> {PHASE_3_TARGET:,}...")
    !python3 scripts/train_v5_graded_curriculum.py --model-preset {MODEL_PRESET} --mix code --total-steps {PHASE_3_TARGET} --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --prompt-len {PROMPT_LEN} --seq-len {SEQ_LEN} --no-elastic-window --max-lr 3.0e-4 --warmup-steps 500 --save-interval 250 --cloud-sync --cloud-sync-interval 100 --amp-dtype {AMP_DTYPE} --optimizer spheroid --deliberation-mode {DELIBERATION_MODE} --schedule-steps {SCHEDULE_STEPS} --min-sampling-p {MIN_SAMPLING_P} --pitfall-prob {PITFALL_PROB} --loss-window-governor --loss-ceiling {LOSS_CEILING} --loss-floor {LOSS_FLOOR} --loss-lr-damping


### 🤖 Option 5E: Mixture 4 - Modern Agentic Workflows & Multi-Turn Tool Execution (`--mix agentic`)
SmolTalk + Hermes Function Calling (`BASH`, `PYTHON`, `CALCULATOR`, `FILE_READ`) + UltraChat multi-turn.

In [ ]:
# Option 5E: Phase 4 - Agentic Tools & Workflows (Target: PHASE_4_TARGET)
curr_step, _ = get_current_model_step(MODEL_PRESET)

if not TRAINING_MODE.startswith("Step-Aware"):
    print("⏩ [Option 5E] Skipped: Single Continuous Run mode is active.")
elif curr_step >= PHASE_4_TARGET:
    print(f"⏩ [Option 5E: Agentic] Step {curr_step:,} >= target {PHASE_4_TARGET:,}. Stage already complete!")
    print("✓ Skipping cell to proceed to next stage...")
else:
    print(f"🚀 [Option 5E: Agentic] Launching training from step {curr_step:,} -> {PHASE_4_TARGET:,}...")
    !python3 scripts/train_v5_graded_curriculum.py --model-preset {MODEL_PRESET} --mix agentic --total-steps {PHASE_4_TARGET} --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --prompt-len {PROMPT_LEN} --seq-len {SEQ_LEN} --no-elastic-window --max-lr 2.5e-4 --warmup-steps 500 --save-interval 250 --cloud-sync --cloud-sync-interval 100 --amp-dtype {AMP_DTYPE} --optimizer spheroid --deliberation-mode {DELIBERATION_MODE} --schedule-steps {SCHEDULE_STEPS} --min-sampling-p {MIN_SAMPLING_P} --pitfall-prob {PITFALL_PROB} --loss-window-governor --loss-ceiling {LOSS_CEILING} --loss-floor {LOSS_FLOOR} --loss-lr-damping


### ⭐ Option 5F: Mixture 5 - Golden Consolidation & Anti-Hallucination Replay (`--mix consolidation`)
Balanced 4-pillar replay + CerebroCognitive-10k (strawberry letter counting, temporal clock reasoning, counterfactual anti-sycophancy).

In [ ]:
# Option 5F: Phase 5 - Golden Consolidation Replay (Target: PHASE_5_TARGET)
curr_step, _ = get_current_model_step(MODEL_PRESET)

if not TRAINING_MODE.startswith("Step-Aware"):
    print("⏩ [Option 5F] Skipped: Single Continuous Run mode is active.")
elif curr_step >= PHASE_5_TARGET:
    print(f"⏩ [Option 5F: Consolidation] Step {curr_step:,} >= target {PHASE_5_TARGET:,}. Stage already complete!")
    print("✓ Skipping cell to proceed to benchmark evaluation...")
else:
    print(f"🚀 [Option 5F: Consolidation] Launching training from step {curr_step:,} -> {PHASE_5_TARGET:,}...")
    !python3 scripts/train_v5_graded_curriculum.py --model-preset {MODEL_PRESET} --mix consolidation --total-steps {PHASE_5_TARGET} --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --prompt-len {PROMPT_LEN} --seq-len {SEQ_LEN} --no-elastic-window --max-lr 2.0e-4 --warmup-steps 500 --save-interval 250 --cloud-sync --cloud-sync-interval 100 --amp-dtype {AMP_DTYPE} --optimizer spheroid --deliberation-mode {DELIBERATION_MODE} --schedule-steps {SCHEDULE_STEPS} --min-sampling-p {MIN_SAMPLING_P} --pitfall-prob {PITFALL_PROB} --loss-window-governor --loss-ceiling {LOSS_CEILING} --loss-floor {LOSS_FLOOR} --loss-lr-damping


## 6. Evaluate on Standardized Cerebro-1000 Benchmark
Evaluates the active model tier against the 1,000 fixed benchmark prompts across Language, Knowledge, Math, Code, and Tools.

In [ ]:
# Run benchmark pulse check on the active model
curr_step, ckpt_loc = get_current_model_step(MODEL_PRESET)

if ckpt_loc and os.path.exists(ckpt_loc):
    print(f"🎯 Running Cerebro-1000 Benchmark Evaluation on {ckpt_loc} (Step {curr_step:,})...")
    !python3 eval/run_benchmark.py --checkpoint {ckpt_loc} --samples 100
else:
    print(f"⚠️ No trained checkpoint found for {MODEL_PRESET} yet. Run training stages first!")
